## Report
### 1. metadata size ✅

* Human: 3395
* Mouse: 4066

### 2. missing values ✅

Title / Summary / Organism：none

### 3. text length ⚠️

|Human | Mouse|
|--|--|
|mean ≈ 132 tokens|mean ≈ 131 tokens|
|65 GSE < 20 tokens|53 GSE < 20 tokens|

1.5-2% has super short text which might cause:
* cannot make valid predictions of disease
* lower cosine similarity

### 4. organism consistency ✅

|Human | Mouse|
|--|--|
|3394 Homo sapiens|4064 Mus musculus|
|1 not found|2 not found|

### 5. mixed species detection ⚠️

Check logic:

```python
text contains "human" AND "mouse"
```

Result: 
* Human: 107
* Mouse: 271

These may represent:
* Xenograft models
* Mouse models of human disease
* Background references to the other species

Further refinement is needed before downstream similarity analysis.

## Code

In [ ]:
import json
import pandas as pd

human_path = "metadata/metadata_human.json"
mouse_path = "metadata/metadata_mouse.json"

with open(human_path, "r") as f:
    human_data = json.load(f)

with open(mouse_path, "r") as f:
    mouse_data = json.load(f)

# 关键修正：from_dict + orient='index'
human_df = pd.DataFrame.from_dict(human_data, orient="index")
mouse_df = pd.DataFrame.from_dict(mouse_data, orient="index")

# 把 GSE ID 变成一列
human_df.reset_index(inplace=True)
mouse_df.reset_index(inplace=True)

human_df.rename(columns={"index": "GSE_ID"}, inplace=True)
mouse_df.rename(columns={"index": "GSE_ID"}, inplace=True)

print("Human shape:", human_df.shape)
print("Mouse shape:", mouse_df.shape)

human_df.head()

if "Accession" in human_df.columns:
    print("Human duplicate GSE:", human_df["Accession"].duplicated().sum())

if "Accession" in mouse_df.columns:
    print("Mouse duplicate GSE:", mouse_df["Accession"].duplicated().sum())

def missing_check(df, name):
    print(f"\n--- {name} ---")
    for col in ["Title", "Summary", "Organism"]:
        if col in df.columns:
            print(f"{col} missing:", df[col].isna().sum())
        else:
            print(f"{col} column not found")

missing_check(human_df, "Human")
missing_check(mouse_df, "Mouse")

def text_length_check(df, name):
    print(f"\n--- {name} text length ---")
    df["text_combined"] = df["Title"].fillna("") + " " + df["Summary"].fillna("")
    df["token_count"] = df["text_combined"].apply(lambda x: len(str(x).split()))
    
    print("Mean tokens:", df["token_count"].mean())
    print("Min tokens:", df["token_count"].min())
    print("Number < 20 tokens:", (df["token_count"] < 20).sum())

text_length_check(human_df, "Human")
text_length_check(mouse_df, "Mouse")

print("\nHuman Organism values:")
print(human_df["Organism"].value_counts())

print("\nMouse Organism values:")
print(mouse_df["Organism"].value_counts())

def mixed_species_check(df, name):
    df["text_lower"] = (df["Title"].fillna("") + " " + df["Summary"].fillna("")).str.lower()
    
    mixed = df[
        df["text_lower"].str.contains("human") &
        df["text_lower"].str.contains("mouse")
    ]
    
    print(f"\n{name} mixed species count:", len(mixed))
    return mixed

mixed_human = mixed_species_check(human_df, "Human")
mixed_mouse = mixed_species_check(mouse_df, "Mouse")

In [3]:
print(human_df.columns)
print(mouse_df.columns)

Index(['GSE100027', 'GSE100040', 'GSE100075', 'GSE100081', 'GSE100092',
       'GSE100099', 'GSE100118', 'GSE100183', 'GSE100206', 'GSE100210',
       ...
       'GSE99825', 'GSE99843', 'GSE99857', 'GSE99867', 'GSE99909', 'GSE99923',
       'GSE99937', 'GSE99944', 'GSE99951', 'GSE99987'],
      dtype='object', length=3395)
Index(['GSE100035', 'GSE100039', 'GSE100067', 'GSE100070', 'GSE100098',
       'GSE100102', 'GSE100106', 'GSE100175', 'GSE100212', 'GSE100217',
       ...
       'GSE99954', 'GSE99957', 'GSE99970', 'GSE99971', 'GSE99972', 'GSE99973',
       'GSE99974', 'GSE99975', 'GSE99977', 'GSE99989'],
      dtype='object', length=4066)
